In [4]:
import jax
import jax.numpy as jnp
import optax

# %run ./optimiser.ipynb

# def loss_fn(params):
#     return -<objective_fn>(params["arg"])
    
# N = 100
# params = {"param_1": jnp.ones(N)}
# fixed_args = ()

# optimise(loss_fn, params, fixed_args, steps=1000) 
# learn_rate = {"param_1": 1e-3, "param_2": 1e-2}

# optimise(loss_fn, params, fixed_args, learn_rate=learn_rate): 


def _normalise_fixed_args(fixed_args):
    if fixed_args is None:
        return ()
    if isinstance(fixed_args, list):
        return tuple(fixed_args)
    if isinstance(fixed_args, tuple):
        return fixed_args
    return (fixed_args,)


def _learning_rate_tree(params, learn_rate=None, default_learning_rate=1e-2):
    if isinstance(params, dict):
        if learn_rate is None:
            learn_rate = {}
        if isinstance(learn_rate, dict):
            return {
                name: _learning_rate_tree(
                    value,
                    learn_rate.get(name),
                    default_learning_rate,
                )
                for name, value in params.items()
            }
    if learn_rate is None:
        learn_rate = default_learning_rate
    return jax.tree_util.tree_map(lambda value: jnp.asarray(learn_rate), params)


def optimise(loss_fn, init_params, fixed_args=(), steps=1000, learn_rate=None, default_learning_rate=1e-2): 
    
    fixed_args = _normalise_fixed_args(fixed_args)
    learning_rates = _learning_rate_tree(
        init_params,
        learn_rate=learn_rate,
        default_learning_rate=default_learning_rate,
    )

    optimizer = optax.scale_by_adam()
    opt_state = optimizer.init(init_params)

    @jax.jit
    def train_step(params, opt_state, fixed_args):
        loss, grads = jax.value_and_grad(loss_fn)(
            params, *fixed_args
        )
    
        updates, opt_state = optimizer.update(grads, opt_state, params)
        updates = jax.tree_util.tree_map(
            lambda update, lr: -lr * update,
            updates,
            learning_rates,
        )
        params = optax.apply_updates(params, updates)
    
        return params, opt_state, -loss

    params = init_params
    for step in range(steps):
        params, opt_state, J = train_step(
            params, opt_state, fixed_args
        )
    return params
